In [5]:
import os
import matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor    

In [6]:
df = pd.read_csv('C:\\Users\\princ\\MLops day 1\\data\\Advertising.csv')
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [7]:
x=df[['TV', 'radio', 'newspaper']]
y=df['sales']

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [8]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [9]:
mlflow.set_experiment("Advertising Sales Prediction")

<Experiment: artifact_location='file:c:/Users/princ/MLops day 1/notebooks/mlruns/1', creation_time=1788640905022, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788640905022, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

In [10]:
from sklearn.metrics import root_mean_squared_error


with mlflow.start_run(run_name="Linear Regression") as run:
    model = LinearRegression()
    model.fit(xtrain, ytrain)

    y_pred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2=r2_score(ytest, y_pred)

# parameters
    mlflow.log_param("model_type", "Linear Regression")
    mlflow.log_param("train_size", 0.2)
    mlflow.log_param("random_state", 56)

# metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)


    

####  now ,we create another run using a Ridge Regression model

In [11]:
with mlflow.start_run(run_name="Ridge Regression"):
    from sklearn.linear_model import Ridge

    model = Ridge(alpha=1.0)
    model.fit(xtrain, ytrain)

    y_pred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2=r2_score(ytest, y_pred)

    # log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # log the matrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # log the artifacts
    mlflow.sklearn.log_model(sk_model = model, name="ridge_reg_model")

Artifacts are teh concrete output files genrated by a run.

Now we use the autologging feturemlflow.

Autologging allows mlflow to aotomatically capture much of the information.

In [12]:
# trun on sciket-learn autologging
mlflow.sklearn.autolog()

In [20]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(xtrain, ytrain)

    test_pred = model.predict(xtest)

    test_mae = mean_absolute_error(ytest, test_pred)
    test_rmse = root_mean_squared_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics
    mlflow.log_metrics({
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    })

    

2026/09/06 13:57:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [21]:
# for registering the model,we require the model URI.
# URI is a unique identifier for the model.

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/9969e34f85824a4fb47a02ee4c28a947/model


##### now lets register th model

In [22]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Advertising_Sales_Model"
)

Registered model 'Advertising_Sales_Model' already exists. Creating a new version of this model...
2026/09/06 13:57:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 9969e34f85824a4fb47a02ee4c28a947 has no artifacts at artifact path 'model', registering model based on models:/m-44ef666cde404de2aaac9b96633fec12 instead
Created version '2' of model 'Advertising_Sales_Model'.


### Model Aliaeses

model aliasing gives a nickname(like"surrent_best" or "production" toa specific version of the registerd model)
instand of typing exact number like version 1,version2,version15,you just use the nickname.
Advertising_sales_model
|
|--version 1
|--version2
|--version3 -champion
|--version4 - chalanger

*champions* means:
the currntly preferred model.

*challanger* means:
a new candiddate being evaluated as a possible replacement.

In [23]:
from mlflow import MlflowClient

client = MlflowClient()
# Assing an alias to a spescific version
# (sets the alias "champion" to version 2 of "fraud_detector")
client.set_registered_model_alias(
    name="Advertising_Sales_Model",
    version=1,
    alias="champion"
)


### Loding a registred model

In [27]:
mlflow.sklearn.log_model(
    model,
    "model",
    registered_model_name="Advertising_sales_model"
)

2026/09/06 14:01:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Advertising_sales_model'.
Created version '1' of model 'Advertising_sales_model'.


In [31]:
import mlflow
import pandas as pd

model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion"
)

new_data = pd.DataFrame({
    "TV": [150, 0],
    "radio": [25, 0],
    "newspaper": [30, 0]
})

predictions = model.predict(new_data)

print(predictions)

[15.3388167  2.9843   ]
